# 06 · Gold layer

Combines both models' test predictions into one table and compares them: where does AraBERT beat the baseline, and can its confidence be trusted?

| | |
|---|---|
| **Input** | Silver test split, `baseline_test_predictions` (from `03`), `arabert_test_predictions` (from `05`) |
| **Output** | `data/gold/test_predictions`: one row per test review with both models' predictions |

## Build the predictions table

Join on `review_id` and add analysis columns: review length in words, whether each model was right, and AraBERT's confidence (the probability of the class it chose).

In [1]:
from config import get_spark, SILVER_PATH, GOLD_PATH
from pyspark.sql import functions as F

spark = get_spark("gold")

silver = spark.read.parquet(SILVER_PATH).filter("split = 'test'")
ar = spark.read.parquet(f"{GOLD_PATH}/arabert_test_predictions")
bl = spark.read.parquet(f"{GOLD_PATH}/baseline_test_predictions")

fact = (silver
    .withColumn("review_id", F.col("review_id").cast("int"))
    .join(ar, "review_id")
    .join(bl, "review_id")
    .withColumn("n_words", F.size(F.split("review_clean", " ")))
    .withColumn("ar_correct", (F.col("ar_pred") == F.col("label")).cast("int"))
    .withColumn("bl_correct", (F.col("bl_pred") == F.col("label")).cast("int"))

    .withColumn("ar_confidence", F.greatest("ar_prob_pos", 1 - F.col("ar_prob_pos")))
    .select("review_id", "hotel_name", "user_type", "room_type", "rating", "label",
            "had_title", "n_words", "review_clean",
            "ar_prob_pos", "ar_pred", "ar_correct", "ar_confidence",
            "bl_prob_pos", "bl_pred", "bl_correct")
)

fact.write.mode("overwrite").parquet(f"{GOLD_PATH}/test_predictions")
fact = spark.read.parquet(f"{GOLD_PATH}/test_predictions")
print("rows:", fact.count())

rows: 10217


## Accuracy by segment

Both models compared within groups of reviews:

- **Length:** both struggle most with short reviews (≤10 words), and that is where AraBERT gains the most (81.7% → 89.8%). With little text, knowing word meaning matters more than counting words.
- **Had a title:** reviews whose score-word title was removed in silver are harder for both models (88.5% vs 94.0% for AraBERT), likely because the text left behind is shorter.
- **Rating:** the extremes (1 and 5) are easiest. Ratings 2 and 4 are mild opinions and hardest, but AraBERT still adds 6-8 points on both.
- **User type:** accuracy is about the same for families, couples and solo travellers, so neither model favours one group. (The one-review group `جناح بغرفتي نوم أسرة توأم` is a room type in the wrong column, a quirk of the source data.)

AraBERT is ahead in every segment with more than one review.

In [5]:
def acc_by(col):
    """Accuracy of both models, split by one column."""
    return (fact.groupBy(col)
            .agg(F.count("*").alias("n"),
                 F.round(F.avg("bl_correct"), 4).alias("baseline_acc"),
                 F.round(F.avg("ar_correct"), 4).alias("arabert_acc"))
            .orderBy(col))

fact = fact.withColumn("length_bucket",
    F.when(F.col("n_words") <= 10, "1_short (<=10)")
     .when(F.col("n_words") <= 40, "2_medium (11-40)")
     .otherwise("3_long (>40)"))

acc_by("length_bucket").show()
acc_by("had_title").show()
acc_by("rating").show()
acc_by("user_type").show(truncate=False)

+----------------+----+------------+-----------+
|   length_bucket|   n|baseline_acc|arabert_acc|
+----------------+----+------------+-----------+
|  1_short (<=10)|3180|      0.8173|     0.8984|
|2_medium (11-40)|5464|      0.8658|      0.922|
|    3_long (>40)|1573|      0.9104|     0.9466|
+----------------+----+------------+-----------+

+---------+----+------------+-----------+
|had_title|   n|baseline_acc|arabert_acc|
+---------+----+------------+-----------+
|    false|6152|      0.8937|     0.9403|
|     true|4065|       0.803|     0.8854|
+---------+----+------------+-----------+

+------+----+------------+-----------+
|rating|   n|baseline_acc|arabert_acc|
+------+----+------------+-----------+
|     1|1363|       0.887|     0.9384|
|     2|3783|      0.8245|     0.8816|
|     4|2538|      0.8136|      0.898|
|     5|2533|      0.9353|     0.9834|
+------+----+------------+-----------+

+-------------------------+----+------------+-----------+
|user_type                |n   |

## Is AraBERT's confidence reliable?

Group predictions by confidence and check the accuracy in each band. Accuracy closely tracks confidence (e.g. predictions made with 70-80% confidence are right 70.8% of the time), so the model is **well calibrated**.

82% of reviews are predicted with ≥90% confidence, and those are right 96.8% of the time. In practice, the ~6% of reviews under 70% confidence could be sent to a person for review.

In [3]:
(fact.withColumn("conf_band", F.floor(F.col("ar_confidence") * 10) / 10)
     .groupBy("conf_band")
     .agg(F.count("*").alias("n"), F.round(F.avg("ar_correct"), 4).alias("arabert_acc"))
     .orderBy("conf_band")
     .show())

+---------+----+-----------+
|conf_band|   n|arabert_acc|
+---------+----+-----------+
|      0.5| 307|     0.5212|
|      0.6| 334|     0.6048|
|      0.7| 441|     0.7075|
|      0.8| 728|     0.7912|
|      0.9|8407|     0.9675|
+---------+----+-----------+

